<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال از TF-IDF برای Keyword Search و Sentence Transformers برای Semantic Search استفاده می‌کنیم. سپس نتایج را با روش Weighted Score ترکیب می‌کنیم.

چگونه نتایج را ترکیب می‌کنیم؟
یک روش ساده، ترکیب امتیازهای دو موتور جستجو با استفاده از وزن‌دهی است:

HybridScore = alpha K + (1-alpha)S

- (K): امتیاز Keyword Search
- (S): امتیاز Semantic Search
- (alpha): وزن Keyword Search

برای مثال:

alpha = 0.4


In [1]:
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer


# ==========================================
# 1. Knowledge Base
# ==========================================

documents = [
    {
        "id": "HR-001",
        "text": "Employees receive 26 annual leave days."
    },
    {
        "id": "HR-002",
        "text": "Leave requests must be approved by the manager."
    },
    {
        "id": "HR-003",
        "text": "Employees have supplementary health insurance."
    },
    {
        "id": "HR-004",
        "text": "Monthly salaries are paid at the end of each month."
    },
    {
        "id": "HR-005",
        "text": "Employees can work remotely two days per week."
    },
    {
        "id": "HR-006",
        "text": "Employees can request unpaid leave."
    }
]

document_texts = [
    doc["text"] for doc in documents
]


# ==========================================
# 2. Keyword Search Setup
# ==========================================

tfidf = TfidfVectorizer()

keyword_vectors = tfidf.fit_transform(
    document_texts
)


# ==========================================
# 3. Semantic Search Setup
# ==========================================

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

semantic_vectors = model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)


# ==========================================
# 4. Hybrid Search
# ==========================================

def hybrid_search(query, top_k=3, alpha=0.4):

    # Keyword Search
    query_keyword_vector = tfidf.transform([query])

    keyword_scores = cosine_similarity(
        query_keyword_vector,
        keyword_vectors
    )[0]

    # Semantic Search
    query_semantic_vector = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(
        query_semantic_vector,
        semantic_vectors
    )[0]

    # Weighted Score Fusion
    hybrid_scores = (
        alpha * keyword_scores
        + (1 - alpha) * semantic_scores
    )

    # Rank Results
    ranked_indices = np.argsort(hybrid_scores)[::-1]

    results = []

    for index in ranked_indices[:top_k]:

        results.append({
            "id": documents[index]["id"],
            "text": documents[index]["text"],
            "keyword_score": float(keyword_scores[index]),
            "semantic_score": float(semantic_scores[index]),
            "hybrid_score": float(hybrid_scores[index])
        })

    return results


# ==========================================
# 5. Test Hybrid Search
# ==========================================

queries = [
    "annual leave days",
    "How much time off can employees take each year?",
    "HR-003 supplementary insurance"
]

for query in queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)

    results = hybrid_search(
        query=query,
        top_k=3,
        alpha=0.4
    )

    for rank, result in enumerate(results, start=1):

        print(f"\nRank: {rank}")
        print(f"ID: {result['id']}")
        print(f"Text: {result['text']}")
        print(f"Keyword Score: {result['keyword_score']:.4f}")
        print(f"Semantic Score: {result['semantic_score']:.4f}")
        print(f"Hybrid Score: {result['hybrid_score']:.4f}")

e:\AI-Engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1645.33it/s]



QUERY: annual leave days

Rank: 1
ID: HR-001
Text: Employees receive 26 annual leave days.
Keyword Score: 0.6912
Semantic Score: 0.7056
Hybrid Score: 0.6999

Rank: 2
ID: HR-006
Text: Employees can request unpaid leave.
Keyword Score: 0.1746
Semantic Score: 0.5127
Hybrid Score: 0.3774

Rank: 3
ID: HR-005
Text: Employees can work remotely two days per week.
Keyword Score: 0.1771
Semantic Score: 0.3663
Hybrid Score: 0.2906

QUERY: How much time off can employees take each year?

Rank: 1
ID: HR-006
Text: Employees can request unpaid leave.
Keyword Score: 0.3846
Semantic Score: 0.5719
Hybrid Score: 0.4970

Rank: 2
ID: HR-005
Text: Employees can work remotely two days per week.
Keyword Score: 0.2782
Semantic Score: 0.5735
Hybrid Score: 0.4554

Rank: 3
ID: HR-001
Text: Employees receive 26 annual leave days.
Keyword Score: 0.1166
Semantic Score: 0.6715
Hybrid Score: 0.4495

QUERY: HR-003 supplementary insurance

Rank: 1
ID: HR-003
Text: Employees have supplementary health insurance.
Keyword 